# BirdCLEF 2026 - data exploration

1. Audio file metadata (durations, sample rates) and the short-clip filter
2. Label tables and dataset overview
3. Taxonomy and name consistency
4. Training clip labels (class balance, ratings)
5. Soundscape labels
6. Example recordings (player, waveform, spectrograms)

Run the cells from top to bottom. Settings (e.g. the short-clip cutoff) are in the imports cell.

## 0. Data download

The cell below downloads the competition data from Kaggle, but only if it is not already in the `data/` folder. Complete these steps once before running it.

**1. Verify your Kaggle account**
1. Sign in (or create an account) at [kaggle.com](https://www.kaggle.com).
2. Go to **Settings → Phone verification, Identity verification** and verify yourself. Kaggle blocks competition data downloads for unverified accounts.

**2. Accept the competition rules**
1. Open the [BirdCLEF 2026 competition](https://www.kaggle.com/competitions/birdclef-2026).
2. Click **Join Competition** / **I Understand and Accept**.

**3. Create a Kaggle API token**
1. Go to **Settings → API** on kaggle.com and create a new token (it starts with `KGAT_`).
2. Copy it right away - Kaggle shows it only once. Never share it or commit it to git.

**4. Make the token available to the notebook**
The token must be stored under the name `KAGGLE_API_TOKEN`. Use one of these options:

**Option A - Colab secrets (when running in Google Colab)**
1. Click the key icon (**Secrets**) in the left sidebar.
2. Add a new secret named `KAGGLE_API_TOKEN` with your token as the value.
3. Switch on **Notebook access** for it.

**Option B - `.env` file (local, VS Code, or Colab)**
1. Copy `.env.example` to `.env` and set `KAGGLE_API_TOKEN=your_token_here`.
2. Running in Colab: upload the `.env` file to the Colab working directory (**Files** panel → upload, usually `/content`). It is lost when the session ends, so upload it again each session.

**5. Run the cell**
Run the next cell. It downloads and extracts the data (~16 GB, so make sure there is enough free disk space). If the data is already there, it only prints the path.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display
from tinytag import TinyTag

# Settings
MIN_DURATION_SECONDS = 2.0  # Training clips shorter than this are dropped.
WINDOW_SECONDS = 5  # Soundscape labels cover 5-second intervals.
N_FFT = 2048  # Spectrogram settings for the example plots.
HOP_LENGTH = 512
N_MELS = 128
PLAYER_MAX_SECONDS = 60  # Longer recordings are cut in the audio player to keep the notebook small.

In [ ]:
COMPETITION = "birdclef-2026"
KAGGLE_TOKEN = "KAGGLE_API_TOKEN"  # name of the Colab secret / .env variable
DATA_ROOT = Path.cwd() / "data"  # git-ignored; kagglehub stores its cache here

# Files and folders that must exist for the data to count as complete
REQUIRED = [
    "train.csv",
    "taxonomy.csv",
    "train_soundscapes_labels.csv",
    "train_audio",
    "train_soundscapes",
]


def is_complete(folder):
    return all((folder / name).exists() for name in REQUIRED)


def find_local_data():
    """Return the folder holding the competition files, or None."""
    for train_csv in DATA_ROOT.rglob("train.csv"):
        if is_complete(train_csv.parent):
            return train_csv.parent
    return None


def load_from_colab_secrets():
    """Read the token from Colab Secrets; return False outside Colab."""
    try:
        from google.colab import userdata
    except ImportError:
        return False

    try:
        value = userdata.get(KAGGLE_TOKEN)
    except Exception:  # secret not defined or notebook access off
        return True
    if value:
        os.environ[KAGGLE_TOKEN] = value.strip()
    return True


def load_from_env_file():
    """Read the token from the nearest .env file (current folder or a parent)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_file = folder / ".env"
        if not env_file.is_file():
            continue
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip().removeprefix("export ").strip()
            value = value.strip().strip("\"'")
            if key == KAGGLE_TOKEN and value:
                os.environ[KAGGLE_TOKEN] = value
        print(f"Loaded .env from {env_file}")
        return True
    return False


DATA_DIR = find_local_data()

if DATA_DIR:
    print(f"Data already available: {DATA_DIR}")
else:
    print(f"Data not found in {DATA_ROOT}, downloading from Kaggle...")

    # Token: 1. Colab secrets, 2. .env file
    load_from_colab_secrets()
    if not os.environ.get(KAGGLE_TOKEN):
        load_from_env_file()
    if not os.environ.get(KAGGLE_TOKEN):
        raise EnvironmentError(
            f"{KAGGLE_TOKEN} not found. Add it as a Colab secret (notebook "
            f"access on) or to a .env file. Working directory: {Path.cwd()}"
        )

    # Keep the kagglehub cache inside the repo (must be set before the import).
    os.environ["KAGGLEHUB_CACHE"] = str(DATA_ROOT)

    try:
        import kagglehub
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
            check=True,
        )
        import kagglehub

    # Authenticate explicitly with the token
    from kagglehub.config import set_kaggle_api_token

    set_kaggle_api_token(os.environ[KAGGLE_TOKEN])

    # Fails here with a clear error if the token is rejected.
    print("Authenticated as:", kagglehub.whoami()["username"])

    # The competition rules must be accepted on kaggle.com (same account).
    DATA_DIR = Path(kagglehub.competition_download(COMPETITION))
    print("Path to competition files:", DATA_DIR)

    if not is_complete(DATA_DIR):
        missing = [n for n in REQUIRED if not (DATA_DIR / n).exists()]
        raise FileNotFoundError(f"Download finished but missing: {missing}")

## 1. Audio file metadata
Reads duration, channels, sample rate and bitrate from each file header (no decoding).
The first run takes a while; the results are cached as CSV files in the data folder.
Filenames are stored relative to their audio folder (`species/file.ogg`, as in `train.csv`),
so the cache works on any machine.

In [ ]:
train_audio_dir = DATA_DIR / "train_audio"
train_soundscapes_dir = DATA_DIR / "train_soundscapes"
train_audio_metadata_csv = DATA_DIR / "train_audio_metadata.csv"
train_soundscapes_metadata_csv = DATA_DIR / "train_soundscapes_metadata.csv"

# All audio files, relative to their folder: "species/file.ogg" and "file.ogg"
train_audio_files = sorted(p.relative_to(train_audio_dir).as_posix() for p in train_audio_dir.glob("*/*.ogg"))
train_soundscapes_files = sorted(p.relative_to(train_soundscapes_dir).as_posix() for p in train_soundscapes_dir.glob("*.ogg"))


def get_audio_metadata(audio_dir: Path, files: list[str], csv_output: Path | None = None) -> pd.DataFrame:
    """Read the header metadata of `files` (paths relative to `audio_dir`)."""
    rows = []
    for f in files:
        audio = TinyTag.get(audio_dir / f)
        rows.append({
            "filename": f,
            "duration": audio.duration,
            "channels": audio.channels,
            "sample_rate": audio.samplerate,
            "bitrate": audio.bitrate,
        })
    metadata_df = pd.DataFrame(rows)
    if csv_output:
        metadata_df.to_csv(csv_output, index=False)
    return metadata_df


def to_relative(path, n_parts: int) -> str:
    """Keep the last `n_parts` components of a (possibly absolute, Windows or POSIX) path."""
    return "/".join(str(path).replace("\\", "/").split("/")[-n_parts:])


def load_audio_metadata(csv_path: Path, audio_dir: Path, files: list[str], n_parts: int) -> pd.DataFrame:
    """Load cached metadata (converting old absolute paths), or compute and cache it."""
    if not csv_path.exists():
        return get_audio_metadata(audio_dir, files, csv_path)
    df = pd.read_csv(csv_path)
    relative = df["filename"].map(lambda p: to_relative(p, n_parts))
    if not relative.equals(df["filename"]):
        df["filename"] = relative
        df.to_csv(csv_path, index=False)  # Rewrite the cache with relative paths.
        print(f"Converted absolute paths to relative in {csv_path.name}")
    return df


train_audio_metadata_df = load_audio_metadata(
    train_audio_metadata_csv, train_audio_dir, train_audio_files, n_parts=2
).set_index("filename")
train_soundscapes_metadata_df = load_audio_metadata(
    train_soundscapes_metadata_csv, train_soundscapes_dir, train_soundscapes_files, n_parts=1
).set_index("filename")

display(train_audio_metadata_df.head())
display(train_soundscapes_metadata_df.head())

In [ ]:
# Sample rates, channels and durations
for name, df in [("Training clips", train_audio_metadata_df), ("Soundscapes", train_soundscapes_metadata_df)]:
    durations = df["duration"]
    print(f"{name}: {len(df):,} files")
    print(f"  sample rates: {sorted(df['sample_rate'].unique().tolist())}")
    print(f"  channels:     {sorted(df['channels'].unique().tolist())}")
    print(f"  duration (s): min {durations.min():.2f}, median {durations.median():.2f}, "
          f"max {durations.max():.2f}, {durations.nunique():,} distinct values")

In [ ]:
# Duration distribution of the training clips (all files, x-axis cut at 120 s)
DURATION_PLOT_LIMIT = 120
durations = train_audio_metadata_df["duration"]

plt.figure(figsize=(10, 5))
plt.hist(durations.clip(upper=DURATION_PLOT_LIMIT), bins=60, edgecolor="black")
plt.axvline(MIN_DURATION_SECONDS, color="red", linestyle="--", label=f"{MIN_DURATION_SECONDS:g} s cutoff")
plt.axvline(WINDOW_SECONDS, color="orange", linestyle="--", label=f"{WINDOW_SECONDS} s window")
plt.title(f"Training clip durations (longer than {DURATION_PLOT_LIMIT} s shown in the last bin)")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of clips")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Clips longer than {DURATION_PLOT_LIMIT} s: {(durations > DURATION_PLOT_LIMIT).sum():,}")

### Drop very short clips
Training clips shorter than `MIN_DURATION_SECONDS` are removed. `train_audio_files` keeps the
remaining clips and is used to filter `train.csv` below.

In [ ]:
is_long_enough = train_audio_metadata_df["duration"] >= MIN_DURATION_SECONDS
train_audio_files = train_audio_metadata_df.index[is_long_enough].tolist()
print(f"Dropped {(~is_long_enough).sum():,} clips shorter than {MIN_DURATION_SECONDS:g} s; "
      f"{len(train_audio_files):,} remain.")

## 2. Label tables
- `taxonomy.csv`: one row per species (class), with its names and animal class.
- `train.csv`: one row per training clip; `primary_label` is the species.
- `train_soundscapes_labels.csv`: species heard in each labeled 5-second soundscape interval.

In [ ]:
taxonomy_df = pd.read_csv(DATA_DIR / "taxonomy.csv")
train_audio_label_df = pd.read_csv(DATA_DIR / "train.csv")
soundscapes_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv").drop_duplicates()

# Keep only the clips that passed the duration filter
rows_before = len(train_audio_label_df)
train_audio_label_df = train_audio_label_df[train_audio_label_df["filename"].isin(train_audio_files)]
print(f"train.csv: {rows_before:,} rows, {len(train_audio_label_df):,} after dropping short clips")

display(taxonomy_df.head())
display(train_audio_label_df.head())
display(soundscapes_df.head())

In [ ]:
# Dataset overview
overview = pd.Series({
    "Training clips (after the duration filter)": len(train_audio_label_df),
    "Soundscape files (all)": len(train_soundscapes_files),
    "Soundscape files with labels": soundscapes_df["filename"].nunique(),
    "Labeled soundscape intervals": len(soundscapes_df),
    "Taxonomy species": len(taxonomy_df),
}, name="Count")
display(overview.to_frame())

## 3. Taxonomy
Checks that species identifiers and names are unique, and that the names used in `train.csv`
match the taxonomy.

In [ ]:
assert taxonomy_df["primary_label"].is_unique, "Primary labels in taxonomy.csv are not unique."
assert taxonomy_df["scientific_name"].is_unique, "Scientific names in taxonomy.csv are not unique."
print(f"Taxonomy: {len(taxonomy_df)} species, identifiers and scientific names are unique.")

# Compare label-name combinations; the outer merge shows those that exist on only one side.
name_columns = ["primary_label", "scientific_name", "common_name", "class_name"]
comparison = taxonomy_df[name_columns].drop_duplicates().merge(
    train_audio_label_df[name_columns].drop_duplicates(),
    on=name_columns, how="outer", indicator=True,
)
side_counts = comparison["_merge"].value_counts()
print(f"Species in both tables:               {side_counts['both']}")
print(f"Only in taxonomy (no training clips): {side_counts['left_only']}")
print(f"Only in train.csv (name mismatch):    {side_counts['right_only']}")
if side_counts["right_only"] > 0:
    display(comparison[comparison["_merge"] == "right_only"])

taxonomy_df = taxonomy_df.set_index("primary_label")

## 4. Training clip labels
Species per animal class (taxonomy vs. training clips), recordings per species, and ratings.

In [ ]:
assert train_audio_label_df["filename"].is_unique, "Filenames in train.csv are not unique."
train_audio_label_df = train_audio_label_df.set_index("filename")

# Species per animal class: how many have training clips?
species_per_class = pd.DataFrame({
    "in_taxonomy": taxonomy_df["class_name"].value_counts(),
    "with_training_clips": train_audio_label_df.drop_duplicates("primary_label")["class_name"].value_counts(),
}).fillna(0).astype(int)
species_per_class["without_training_clips"] = (
    species_per_class["in_taxonomy"] - species_per_class["with_training_clips"]
)
display(species_per_class.sort_values("in_taxonomy", ascending=False))

# Recordings per species
clips_per_species = train_audio_label_df["primary_label"].value_counts()
taxonomy_df["training_audio_files_count"] = clips_per_species.reindex(taxonomy_df.index, fill_value=0)

print("Species with the most and the fewest training clips:")
species_counts = train_audio_label_df.value_counts(["primary_label", "common_name", "class_name"])
display(pd.concat([species_counts.head(5), species_counts.tail(5)]).to_frame("clips"))

print("Training clips per species, by animal class:")
with_clips = taxonomy_df[taxonomy_df["training_audio_files_count"] > 0]
display(
    with_clips.groupby("class_name")["training_audio_files_count"]
    .agg(species="count", min="min", median="median", mean="mean", max="max")
    .round(1)
    .sort_values("species", ascending=False)
)

In [ ]:
# Distributions: recordings per species and recording ratings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(clips_per_species.values, bins=40, edgecolor="black")
axes[0].set_title("Training clips per species")
axes[0].set_xlabel("Clips per species")
axes[0].set_ylabel("Number of species")

ratings = pd.to_numeric(train_audio_label_df["rating"], errors="coerce")
axes[1].hist(ratings.dropna(), bins=20, edgecolor="black")
axes[1].set_title("Recording ratings")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Number of clips")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Soundscape labels
Each row labels one 5-second interval of a 60-second soundscape with the species heard in it
(`;`-separated). Not every interval of a file is labeled.

In [ ]:
def soundscape_labels_to_list(labels):
    """Convert a ';'-separated label string to a list of labels."""
    if pd.isna(labels):
        return []
    return [label.strip() for label in labels.split(";")]


soundscapes_df["labels"] = soundscapes_df["primary_label"].map(soundscape_labels_to_list)

# Labeled intervals per file (a fully labeled 60 s file has 12)
intervals_per_file = soundscapes_df["filename"].value_counts()
print(f"{len(intervals_per_file)} labeled soundscape files; labeled intervals per file: "
      f"min {intervals_per_file.min()}, median {intervals_per_file.median():g}, max {intervals_per_file.max()}")

if soundscapes_df.duplicated(subset=["filename", "start", "end"]).any():
    print("WARNING: some (filename, start, end) intervals appear in more than one row.")
else:
    print("Every (filename, start, end) interval appears once.")

# One row per (interval, species)
soundscape_species = (
    soundscapes_df[["filename", "labels"]]
    .explode("labels")
    .rename(columns={"labels": "label"})
    .dropna()
)
soundscape_species["class_name"] = soundscape_species["label"].map(taxonomy_df["class_name"])

taxonomy_df["soundscape_label_count"] = (
    soundscape_species["label"].value_counts().reindex(taxonomy_df.index, fill_value=0)
)
taxonomy_df["soundscape_file_count"] = (
    soundscape_species.groupby("label")["filename"].nunique().reindex(taxonomy_df.index, fill_value=0)
)

print("\nSoundscape coverage per animal class:")
display(
    soundscape_species.groupby("class_name")
    .agg(species=("label", "nunique"), files=("filename", "nunique"), labeled_intervals=("label", "size"))
    .sort_values("files", ascending=False)
)

only_in_soundscapes = taxonomy_df[
    (taxonomy_df["training_audio_files_count"] == 0) & (taxonomy_df["soundscape_label_count"] > 0)
]
print(f"Species with no training clips that appear in soundscapes: {len(only_in_soundscapes)}")
display(only_in_soundscapes[["common_name", "class_name", "soundscape_label_count", "soundscape_file_count"]])

In [ ]:
# Number of species per labeled soundscape interval
species_per_interval = soundscapes_df["labels"].map(len).value_counts().sort_index()

plt.figure(figsize=(9, 5))
plt.bar(species_per_interval.index, species_per_interval.values, edgecolor="black")
plt.xticks(species_per_interval.index)
plt.title("Species per labeled soundscape interval")
plt.xlabel("Number of species")
plt.ylabel("Number of intervals")
plt.tight_layout()
plt.show()

## 6. Example recordings
Audio player, waveform, STFT spectrogram and mel spectrogram. Change the number in each cell
to look at a different recording.

In [ ]:
def plot_recording(y, sr, title, figsize=(14, 10)):
    """Plot the waveform, STFT spectrogram and mel spectrogram of a recording; return the axes."""
    stft_db = librosa.amplitude_to_db(
        np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)), ref=np.max
    )
    mel_db = librosa.power_to_db(
        librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS),
        ref=np.max,
    )
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)

    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Waveform - {title}")
    axes[0].set_ylabel("Amplitude")

    img = librosa.display.specshow(stft_db, sr=sr, hop_length=HOP_LENGTH, x_axis="time", y_axis="hz", ax=axes[1])
    axes[1].set_title("STFT spectrogram (dB)")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")

    img = librosa.display.specshow(mel_db, sr=sr, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel", ax=axes[2])
    axes[2].set_title(f"Mel spectrogram (dB, {N_MELS} bands)")
    fig.colorbar(img, ax=axes[2], format="%+2.0f dB")
    # Hidden colorbar: keeps the waveform panel aligned with the spectrograms.
    fig.colorbar(img, ax=axes[0]).ax.set_visible(False)
    return fig, axes


def play(y, sr):
    """Show an audio player for (at most) the first PLAYER_MAX_SECONDS seconds."""
    display(Audio(y[: int(PLAYER_MAX_SECONDS * sr)], rate=sr))

### Training clip

In [ ]:
sample_number = 1956  # Index into train_audio_files; change it to see another clip.

sample_file = train_audio_files[sample_number]
label_row = train_audio_label_df.loc[sample_file]
print(f"File: {sample_file}")
print(f"Species: {label_row['common_name']} ({label_row['scientific_name']}), "
      f"label {label_row['primary_label']}, class {label_row['class_name']}")
display(train_audio_metadata_df.loc[[sample_file]])

y, sr = librosa.load(train_audio_dir / sample_file, sr=None)  # Native sample rate
play(y, sr)
plot_recording(y, sr, sample_file, figsize=(12, 10))
plt.tight_layout()
plt.show()

### Soundscape with its interval labels
Dashed lines mark the labeled 5-second intervals; the species labels are written at the top
of each interval.

In [ ]:
def time_to_seconds(t):
    """Convert 'HH:MM:SS' to seconds."""
    h, m, s = (int(x) for x in t.split(":"))
    return h * 3600 + m * 60 + s


soundscape_number = 0  # Index into the labeled soundscape files; change it to see another one.

labeled_soundscape_files = sorted(soundscapes_df["filename"].unique())
soundscape_file = labeled_soundscape_files[soundscape_number]
sections = soundscapes_df[soundscapes_df["filename"] == soundscape_file].sort_values("start")

print(f"File: {soundscape_file}")
print("Labels per interval:")
for row in sections.itertuples():
    print(f"  {row.start} - {row.end}: {row.labels}")
print("Species:")
for label in sorted({label for labels in sections["labels"] for label in labels}):
    name = taxonomy_df.loc[label, "common_name"] if label in taxonomy_df.index else "?"
    print(f"  {label}: {name}")

y, sr = librosa.load(train_soundscapes_dir / soundscape_file, sr=None)  # Native sample rate
play(y, sr)
fig, axes = plot_recording(y, sr, soundscape_file, figsize=(16, 12))

# Mark the labeled intervals on every panel and write their species on the spectrograms
for ax in axes:
    for row in sections.itertuples():
        for boundary in (time_to_seconds(row.start), time_to_seconds(row.end)):
            ax.axvline(boundary, color="white" if ax is not axes[0] else "gray", linestyle="--", linewidth=0.8)
for ax in axes[1:]:
    ymax = ax.get_ylim()[1]
    for row in sections.itertuples():
        start, end = time_to_seconds(row.start), time_to_seconds(row.end)
        ax.text(
            (start + end) / 2, ymax * 0.98, "\n".join(row.labels),
            ha="center", va="top", fontsize=7, color="white",
            bbox=dict(facecolor="black", alpha=0.5, pad=1.5, edgecolor="none"),
        )
plt.tight_layout()
plt.show()